# ethical wall rag v2: reranker rerun

The first v2 run gave the reranker only the chunk text while the dense index embeds title + text, which pushed it toward first pages. This reruns just the two reranker rows of the retrieval comparison after the fix.

Runtime: A100. Results go to a fixed Drive folder, so if the runtime drops, run all again and it picks up where it stopped.

In [1]:
!pip install -q "git+https://github.com/arihaaant/ethical-wall-rag.git@v2"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 8.8 MB/s eta 0:00:00


In [2]:
import json, time
from google.colab import drive

drive.mount('/content/drive')
out = '/content/drive/MyDrive/ethical-wall-rag/v2-rerank-fix'

Mounted at /content/drive


In [3]:
from ethicalwall import run_v2

t0 = time.time()
run_v2.main(['--retrieval-only', '--retrievers', 'bge_rerank', '--out', out])
print(f'{(time.time() - t0) / 60:.1f} min')

README.md:   0%|          | 0.00/15.5k [00:00<?, ?B/s]

cuad-qa.py:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

default/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 4.32MB            

default/train/0000.parquet: downloading bytes:           |  0.00B            

default/train/0001.parquet: reconstructing file:   0%|          |  0.00B / 4.08MB            

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

bge_rerank category hit@5 0.535
bge_rerank question hit@5 0.463
24.6 min


In [4]:
import pandas as pd

s = json.load(open(f'{out}/summary_retrieval.json'))
pct = lambda d: round(100 * d['rate'], 1)
pd.DataFrame({name: {'hit@5 all': pct(v['all']), 'clauses': pct(v['clauses']),
                     'metadata': pct(v['metadata']), 'top1': pct(v['top1'])}
              for name, v in s.items()}).T

,hit@5 all,clauses,metadata,top1
bge_rerank/category,53.5,45.6,74.5,25.2
bge_rerank/question,46.3,34.0,78.8,23.9
